# Explorations Solutions of first ANN notebook

## Original code below

In [1]:
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)

data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print(f"X_train_tensor shape: {X_train_tensor.shape}") 
print(f"y_train_tensor shape: {y_train_tensor.shape}") 


class BinaryClassifier(nn.Module):
    def __init__(self, num_features):
        super(BinaryClassifier, self).__init__()
        self.linear = nn.Linear(num_features, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred

num_features = X_train.shape[1]
model = BinaryClassifier(num_features)
print("\n--- Model Architecture ---")
print(model)

with torch.no_grad(): 
    y_pred_test = model(X_test_tensor[:5])

print("\n--- Untrained Model Predictions ---")
print(f"Input shape: {X_test_tensor[:5].shape}")
print(f"Prediction shape: {y_pred_test.shape}")
print("Predictions (probabilities):")
print(y_pred_test.squeeze())
print("\nActual labels:")
print(y_test_tensor[:5].squeeze())

# The predictions are close to 0.5 because the model is initialized randomly.
# It hasn't learned anything yet!

X_train_tensor shape: torch.Size([455, 30])
y_train_tensor shape: torch.Size([455, 1])

--- Model Architecture ---
BinaryClassifier(
  (linear): Linear(in_features=30, out_features=1, bias=True)
)

--- Untrained Model Predictions ---
Input shape: torch.Size([5, 30])
Prediction shape: torch.Size([5, 1])
Predictions (probabilities):
tensor([0.4829, 0.7515, 0.5601, 0.4203, 0.5035])

Actual labels:
tensor([1., 0., 0., 1., 1.])


## Multi-Level Explorations

### Beginner

In [2]:
# Simple `model.parameters()` was not working before.
# I randomly used next().
# And then it worked.

# MMaybe it is because generators are used to reduce memory consumption.
# And using next aloows to get only when caleed or something. IDK
next(model.parameters())

Parameter containing:
tensor([[ 0.1396,  0.1515, -0.0428,  0.1677, -0.0400,  0.0368, -0.0889,  0.1072,
          0.1609, -0.1339,  0.1587,  0.0342,  0.1349,  0.0247,  0.0880, -0.0258,
          0.1407,  0.0270, -0.0852,  0.0465, -0.0841, -0.0214, -0.0742,  0.1211,
         -0.1441, -0.0842, -0.0516, -0.1098,  0.0172, -0.1803]],
       requires_grad=True)

### Intermediate

In [3]:
# Adding a hidden layer to BinaryClassifier
class BinaryClassifierV2(nn.Module):
    def __init__(self, num_features):
        super(BinaryClassifierV2, self).__init__()
        self.layer1 = nn.Linear(num_features, 10)
        self.layer2 = nn.Linear(10, 1)

    def forward(self, x):
        y_pred = self.layer1(x)
        y_pred = torch.relu(y_pred)
        y_pred = self.layer2(y_pred)
        y_pred = torch.sigmoid(y_pred)
        return y_pred


clf = BinaryClassifierV2(num_features)

print(clf)


BinaryClassifierV2(
  (layer1): Linear(in_features=30, out_features=10, bias=True)
  (layer2): Linear(in_features=10, out_features=1, bias=True)
)


### Advanced

In [4]:
# Hooks

def activation_printer_hook(module, input, output):
    print(f"--- Hook on {module.__class__.__name__} ---")
    print(f"Input type: {type(input)}, Input length: {len(input)}")
    print(f"Output shape: {output.shape}")
    print("---------------------------------")

model = BinaryClassifierV2(num_features)

# Register the hook on the first layer
hook_handle = model.layer1.register_forward_hook(activation_printer_hook)

# Perform a forward pass
output = model(X_train_tensor)

--- Hook on Linear ---
Input type: <class 'tuple'>, Input length: 1
Output shape: torch.Size([455, 10])
---------------------------------


# Dataset specific Excercises

In [5]:
from sklearn.datasets import make_classification
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generate synthetic data
X, y = make_classification(
    n_samples=1000,
    n_features=15,
    n_classes=2,
    random_state=42
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Convert to tensors
X_train = torch.tensor(X_train, device=device, dtype=torch.float32)
X_test = torch.tensor(X_test, device=device, dtype=torch.float32)
y_train = torch.tensor(y_train, device=device, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, device=device, dtype=torch.float32).unsqueeze(1)

# Model
class BinaryClassifier(nn.Module):
    def __init__(self, num_features: int):
        super(BinaryClassifier, self).__init__()
        self.fc1 = nn.Linear(num_features, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.sigmoid(x)
        return x

num_features = X_train.shape[1]
bclf = BinaryClassifier(num_features=num_features).to(device)

# Test forward pass
with torch.no_grad():
    out = bclf(X_train[:5])
print(out)


tensor([[0.3717],
        [0.5357],
        [0.6016],
        [0.4219],
        [0.5844]], device='cuda:0')


### Modification

In [6]:
torch.round(out)

tensor([[0.],
        [1.],
        [1.],
        [0.],
        [1.]], device='cuda:0')

### Creation

In [10]:
import torch
from sklearn.datasets import _california_housing
from sklearn.model_selection import train_test_split
import  torch.nn as nn
X,y = _california_housing.fetch_california_housing(return_X_y=True)
X.shape, y.shape

((20640, 8), (20640,))

In [20]:
device =  torch.device("cuda" if torch.cuda.is_available else 'cpu')

class RegANN(nn.Module):
    def __init__(self, cali_features):
        super(RegANN, self).__init__()

        self.fc1 = nn.Linear(cali_features, 512)
        self.activation1 = nn.ReLU()
        self.fc2 = nn.Linear(512, 1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.activation1(x)
        x = self.fc2(x)
        return x
    

X_tensor = torch.tensor(X, dtype = torch.float32, device=device)
y_tensor = torch.tensor(y, dtype = torch.float32, device=device).view(-1,1)


X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)


cali_features = X_train.shape[1]



model = RegANN(cali_features).to(device)
model.forward(X_train[:5])

    

tensor([[-166.0262],
        [-105.4550],
        [ -73.1929],
        [-109.7756],
        [ -77.0540]], device='cuda:0', grad_fn=<AddmmBackward0>)